In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO

import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

from tqdm import tqdm
from torchvision import models, Module
from torch.utils.data import DataLoader, random_split
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt
import random
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import InterpolationMode

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import os

import os
import glob
from tqdm import tqdm

# Define dataset path
dataset_path = os.path.join(path, "dataset", "images")
# Get all image paths
train_image_paths = glob.glob(f"{dataset_path}/*.jpg")

# Define dataset path
dataset_path = os.path.join(path, "dataset", "masks")
# Get all image paths
valid_image_paths = glob.glob(f"{dataset_path}/*.png")



class FloodSegmentationDataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.metadata = csv_file
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        image = Image.open(self.root_dir[idx]).convert("RGB")
        mask = Image.open(self.metadata[idx]).convert("L")  # Convert mask to grayscale (1 channel = binary segmentation)

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Replace mask values with remapped values
        mask = remap_mask(mask)

        return image, mask


In [ ]:
image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),
])

dataset = FloodSegmentationDataset(train_image_paths, valid_image_paths, image_transforms, mask_transforms)

train_len = int(len(dataset) * 0.8)
val_len = len(dataset) - train_len

train_dataset, val_dataset = random_split(dataset, [train_len, val_len])


train_loader = DataLoader(train_dataset, 32, True)
test_loader = DataLoader(val_dataset, 32, False)

In [ ]:
img, mask = next(iter(test_loader))

fig, ax = plt.subplots(5,2, figsize=(10,15))

for i in range(5):
    ax[i,0].imshow(img[i].permute(1,2,0))
    ax[i,1].imshow(mask[i].squeeze(0), cmap='gray')

    print(torch.unique(mask[1].squeeze(0)))

In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cuda"
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,
).to(device)

In [ ]:

def train(model, optimizer, criterion, train_loader, device):
    model.train()
    total_loss = 0.0

    for img, mask in tqdm(train_loader):
        img = img.to(device)
        mask = mask.squeeze(dim=1).to(device)

        outputs = model(img)

        loss = criterion(outputs, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

def validate(model, criterion, test_loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for img, mask in tqdm(test_loader):
            img = img.to(device)
            mask = mask.squeeze(dim=1).to(device)

            outputs = model(img)
            loss = criterion(outputs, mask)
            total_loss += loss.item()

    return total_loss / len(test_loader)

In [ ]:
# TO DO

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

train_losses = []
val_losses = []

e = 20

# Training Loop
for epoch in range(e):
    train_loss = train(model, optimizer, criterion, train_loader, device)
    val_loss = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{e}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")



In [ ]:
plt.plot(range(1, e+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, e+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
# TO DO
model.eval()

test_samples = random.sample(range(len(val_dataset)), 5)

for i in test_samples:
  img, mask = val_dataset[i]

  with torch.no_grad():
    pred_mask = model(img.unsqueeze(0).to(device))
    pred_mask = torch.argmax(pred_mask, dim=1).squeeze().cpu()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))


    axes[0].imshow(img.permute(1,2,0))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze())
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask)
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()